# Synthetic dataset families

Five configurable graph families, with one 128-point example each. These are
**direct similarity graphs**, not feature vectors passed through standardization
or a Gaussian kernel. All algorithms can consume their `similarities` arrays.

The examples share seed 4234 and noise standard deviation 0.02. Node ordering
is shuffled to avoid feeding algorithms a pre-sorted hierarchy. For inspection,
heatmaps reverse this shuffle. No planted labels are inputs to clustering.

- Balanced nested: three binary levels, eight equally sized leaf groups.
- Unbalanced nested: three binary levels, with 75:25 splits where feasible.
- Chains and bridges: four dense groups linked sequentially by six bridge
  points per connection; only consecutive bridge points have strong affinity.
- Overlapping: four groups; 25% of points have equal membership in two groups.
- Core-periphery: a 20% central core connected to four peripheral groups.

Noise is added independently to each unordered pair and mirrored, then clipped
to [0,1]. Diagonals are zero. `planted_similarities` stores the pre-noise graph.
Ground-truth memberships describe the construction, not a certificate of OPT.


In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import hierarchical_clustering
from hierarchical_clustering.synthetic import generate_synthetic, save_synthetic_families

ROOT = Path(hierarchical_clustering.__file__).resolve().parents[2]
manifest = save_synthetic_families(ROOT, n=128, seed=4234, noise=0.02)
synthetic = {}
for entry in manifest:
    name = entry["family"]
    with np.load(ROOT / "data" / "processed" / "synthetic" / f"{name}.npz",
                 allow_pickle=False) as saved:
        synthetic[name] = {key: saved[key] for key in saved.files}
pd.DataFrame(manifest)[["family", "n", "seed", "noise_std", "parameters"]]


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 9), constrained_layout=True)
for ax, (name, arrays) in zip(axes.flat, synthetic.items()):
    order = arrays["display_order"]
    image = ax.imshow(arrays["similarities"][np.ix_(order, order)],
                      vmin=0, vmax=1, cmap="viridis", interpolation="nearest")
    ax.set(title=name.replace("_", " ").title(), xlabel="Point", ylabel="Point")
axes.flat[-1].axis("off")
fig.colorbar(image, ax=list(axes.flat[:5]), label="Similarity", shrink=0.8)
output = ROOT / "results" / "figures"
output.mkdir(parents=True, exist_ok=True)
fig.savefig(output / "synthetic_families.png", dpi=150)
plt.show()


## Creating variants

Call `generate_synthetic` with a different seed for an independent instance,
or adjust family-specific parameters. Examples:

```python
arrays, metadata = generate_synthetic("unbalanced_nested", n=256,
                                     split_fraction=0.8, noise=0.05, seed=7)
arrays, metadata = generate_synthetic("overlapping", overlap_fraction=0.4, seed=7)
```

Pass `arrays["similarities"]` to any of our graph clustering functions.
The feature-based Iris/Wine preprocessing is not applied to these graphs.
The synthetic benchmark below uses all six methods on the five saved graphs.
If you regenerate graphs with new settings, rerun the benchmark before comparing scores.


## Six-method benchmark

One generated instance per family, with fixed generation seed 4234. Single,
complete, average, and spectral run once; random trees use 100 algorithm seeds
and basic-SDP Max-Uncut uses 20 (32 Gaussian candidates per node).
This is a Wu-inspired heuristic without hierarchy or conditioning; no 0.8732 guarantee.

The ratios use `(n-2)*W`, an upper bound on optimal revenue. Error bars show
one sample standard deviation over algorithm seeds, not variation across
independent generated datasets. Deterministic methods have no error bars.
These results do not establish which method is best across a whole family.


In [ ]:
from hierarchical_clustering.synthetic_experiments import run_synthetic_experiments

# Run the new benchmark if no basic-SDP results have been saved.
if not (ROOT / "results" / "synthetic_sdp" / "scores.csv").exists():
    run_synthetic_experiments(ROOT)
synthetic_scores = pd.read_csv(ROOT / "results" / "synthetic_sdp" / "scores.csv")
synthetic_summary = pd.read_csv(ROOT / "results" / "synthetic_sdp" / "summary.csv")
method_order = ["single", "complete", "average", "random", "spectral", "max_uncut_sdp"]
ratio_table = synthetic_summary.pivot(
    index="family", columns="method", values="revenue_over_upper_bound_mean"
)[method_order]
ratio_table


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8), constrained_layout=True)
for ax, family in zip(axes.flat, synthetic):
    subset = synthetic_summary[synthetic_summary["family"] == family].set_index("method").loc[method_order]
    ax.bar(method_order, subset["revenue_over_upper_bound_mean"],
           yerr=subset["revenue_over_upper_bound_std"].fillna(0), capsize=3)
    ax.axhline(1 / 3, color="black", linestyle="--", linewidth=1)
    ax.set(title=family.replace("_", " ").title(), ylabel="Revenue / upper bound", ylim=(0, 1))
    ax.tick_params(axis="x", rotation=35)
axes.flat[-1].axis("off")
fig.savefig(ROOT / "results" / "figures" / "synthetic_revenue_comparison_sdp.png", dpi=150)
plt.show()


In [ ]:
synthetic_summary[["family", "method", "runs", "cost_mean", "cost_std",
                   "revenue_mean", "revenue_std", "tree_seconds_mean"]]
